# DSFB-DSCD results replay (zip upload + local outputs)

This notebook replays DSCD results generated by the Rust crate.

Generate local outputs (optional):

```bash
cargo run --release -p dsfb-dscd -- --num-events 2048 --tau-steps 201
```

Interactive dropdown controls for `num-events`, `tau-steps`, and graph preview edge count are provided in the first code cell.

Then either:
- place outputs under `output-dsfb-dscd/<timestamp>/`, or
- upload a `.zip` from your local machine (Colab picker supported).

Accepted zip layouts:
- `output-dsfb-dscd/<timestamp>/...`
- `<timestamp>/...` (contains either `threshold_sweep.csv` or `threshold_scaling_summary.csv` + `threshold_curve_N_*.csv`, plus `finite_size_scaling.csv`)



In [ ]:
from pathlib import Path
from typing import List, Optional
import os
import shutil
import subprocess
import zipfile
import io
import csv

AUTO_ENFORCE_LATEST_MAIN = False
DSFB_MAIN_REPO = 'https://github.com/infinityabundance/dsfb.git'

def _run_git(repo_root: Path, *args: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        ['git', *args],
        cwd=repo_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )


def _git_repo_root() -> Optional[Path]:
    if shutil.which('git') is None:
        return None
    probe = subprocess.run(
        ['git', 'rev-parse', '--show-toplevel'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    if probe.returncode != 0:
        return None
    return Path(probe.stdout.strip()).resolve()


def _in_colab_runtime() -> bool:
    return Path('/content').exists()


def ensure_latest_main() -> Path:
    if not AUTO_ENFORCE_LATEST_MAIN:
        repo_root = _git_repo_root()
        return repo_root if repo_root is not None else Path.cwd().resolve()

    repo_root = _git_repo_root()
    if repo_root is None:
        if not _in_colab_runtime():
            raise RuntimeError(
                'git repo not detected. Open this notebook from a dsfb git checkout of main.'
            )

        colab_repo = Path('/content/dsfb')
        if colab_repo.exists() and not (colab_repo / '.git').exists():
            raise RuntimeError(
                '/content/dsfb exists but is not a git repository. Remove it and rerun.'
            )
        if not colab_repo.exists():
            print(f'Cloning DSFB main into {colab_repo} ...')
            clone = subprocess.run(
                ['git', 'clone', '--depth', '1', '--branch', 'main', DSFB_MAIN_REPO, str(colab_repo)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
            )
            if clone.returncode != 0:
                raise RuntimeError(f'Failed to clone DSFB main:\n{clone.stderr.strip()}')
        repo_root = colab_repo.resolve()
        os.chdir(repo_root)
        print(f'Using cloned repository at {repo_root}')

    fetch = _run_git(repo_root, 'fetch', 'origin', 'main')
    if fetch.returncode != 0:
        raise RuntimeError(f'Failed to fetch origin/main:\n{fetch.stderr.strip()}')

    local = _run_git(repo_root, 'rev-parse', 'HEAD')
    remote = _run_git(repo_root, 'rev-parse', 'origin/main')
    if local.returncode != 0 or remote.returncode != 0:
        raise RuntimeError('Failed to resolve HEAD/origin/main SHA.')

    local_sha = local.stdout.strip()
    remote_sha = remote.stdout.strip()
    print(f'Repo HEAD={local_sha[:12]} origin/main={remote_sha[:12]}')
    if local_sha == remote_sha:
        return repo_root

    dirty = _run_git(repo_root, 'status', '--porcelain')
    if dirty.returncode != 0:
        print('Warning: repo status check failed; continuing with current checkout for replay mode.')
        return repo_root
    if dirty.stdout.strip():
        print('Warning: repo is behind origin/main and has local changes; using current checkout for replay mode.')
        return repo_root

    checkout = _run_git(repo_root, 'checkout', 'main')
    if checkout.returncode != 0:
        raise RuntimeError(f'Failed to checkout main:\n{checkout.stderr.strip()}')
    pull = _run_git(repo_root, 'pull', '--ff-only', 'origin', 'main')
    if pull.returncode != 0:
        raise RuntimeError(f'Failed to fast-forward main:\n{pull.stderr.strip()}')

    updated = _run_git(repo_root, 'rev-parse', 'HEAD').stdout.strip()
    print(f'Repository updated to latest main ({updated[:12]}). Continuing in this runtime.')
    return repo_root


REPO_ROOT = ensure_latest_main()

try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception:
    widgets = None
    display = None

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None


# Optional explicit overrides
RUN_DIR = None  # e.g. Path('/content/output-dsfb-dscd/20260302_000000')
OUTPUT_ROOT = None  # e.g. Path('/content/output-dsfb-dscd')
RESULTS_ZIP_PATH = None  # e.g. Path('/content/dsfb-dscd-results.zip')

TAU_STEPS_OPTIONS = [201, 402, 804, 1608]
EVENT_COUNT_OPTIONS = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144]
EDGE_PREVIEW_OPTIONS = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]

DEFAULT_TAU_STEPS = 201
DEFAULT_NUM_EVENTS = 512
DEFAULT_EDGE_PREVIEW = 512

tau_steps_dropdown = None
num_events_dropdown = None
edge_preview_dropdown = None

if widgets is not None and display is not None:
    tau_steps_dropdown = widgets.Dropdown(
        options=TAU_STEPS_OPTIONS,
        value=DEFAULT_TAU_STEPS,
        description='tau steps',
    )
    num_events_dropdown = widgets.Dropdown(
        options=EVENT_COUNT_OPTIONS,
        value=DEFAULT_NUM_EVENTS,
        description='events',
    )
    edge_preview_dropdown = widgets.Dropdown(
        options=EDGE_PREVIEW_OPTIONS,
        value=DEFAULT_EDGE_PREVIEW,
        description='edges',
    )
    display(widgets.HBox([num_events_dropdown, tau_steps_dropdown, edge_preview_dropdown]))
else:
    print('ipywidgets unavailable; using default controls: '
          f'events={DEFAULT_NUM_EVENTS}, tau_steps={DEFAULT_TAU_STEPS}, edges={DEFAULT_EDGE_PREVIEW}')


def selected_colab_params() -> dict:
    return {
        'num_events': int(num_events_dropdown.value) if num_events_dropdown is not None else DEFAULT_NUM_EVENTS,
        'tau_steps': int(tau_steps_dropdown.value) if tau_steps_dropdown is not None else DEFAULT_TAU_STEPS,
        'edge_preview_limit': int(edge_preview_dropdown.value) if edge_preview_dropdown is not None else DEFAULT_EDGE_PREVIEW,
    }


def print_sweep_command_from_controls() -> None:
    params = selected_colab_params()
    cmd = (
        'cargo run --release -p dsfb-dscd -- '
        f"--num-events {params['num_events']} --tau-steps {params['tau_steps']}"
    )
    print('Selected controls:', params)
    print('Suggested local generation command:')
    print(cmd)


def run_sweep_from_controls() -> None:
    params = selected_colab_params()
    cmd = [
        'cargo', 'run', '--release', '-p', 'dsfb-dscd', '--',
        '--num-events', str(params['num_events']),
        '--tau-steps', str(params['tau_steps']),
    ]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)


print_sweep_command_from_controls()

AUTO_BOOTSTRAP_SWEEP = False

def _cargo_available() -> bool:
    return shutil.which('cargo') is not None


def ensure_cargo_available() -> bool:
    if _cargo_available():
        return True
    if not _in_colab_runtime():
        return False

    print('cargo not found; installing Rust toolchain with rustup...')
    install = subprocess.run(
        ['bash', '-lc', 'curl https://sh.rustup.rs -sSf | sh -s -- -y'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    if install.returncode != 0:
        print(install.stderr.strip())
        return False

    cargo_bin = Path.home() / '.cargo' / 'bin'
    os.environ['PATH'] = f"{cargo_bin}:{os.environ.get('PATH', '')}"
    return _cargo_available()


def _is_sweep_run_dir(path: Path) -> bool:
    return (path / 'threshold_sweep.csv').exists()


def _is_scaling_run_dir(path: Path) -> bool:
    return (
        (path / 'threshold_scaling_summary.csv').exists()
        and any(path.glob('threshold_curve_N_*.csv'))
    )


def _is_run_dir(path: Path) -> bool:
    return path.is_dir() and (_is_sweep_run_dir(path) or _is_scaling_run_dir(path))


def _unique_paths(paths: List[Path]) -> List[Path]:
    out: List[Path] = []
    seen = set()
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        out.append(path)
    return out


def _default_unpack_root() -> Path:
    if Path('/content').exists():
        return Path('/content/output-dsfb-dscd')
    return Path.cwd().resolve() / 'output-dsfb-dscd'


def _candidate_output_roots() -> List[Path]:
    roots: List[Path] = []

    if OUTPUT_ROOT is not None:
        roots.append(Path(OUTPUT_ROOT).expanduser().resolve())

    cwd = Path.cwd().resolve()
    roots.extend([
        cwd / 'output-dsfb-dscd',
        *[parent / 'output-dsfb-dscd' for parent in cwd.parents],
        Path.home() / 'output-dsfb-dscd',
        Path('/content/output-dsfb-dscd'),
        Path('/content/dsfb/output-dsfb-dscd'),
        Path('/content/dsfb/crates/dsfb-dscd/output-dsfb-dscd'),
        Path('/workspace/output-dsfb-dscd'),
        Path('/workspaces/output-dsfb-dscd'),
        Path('/tmp/output-dsfb-dscd'),
    ])

    return _unique_paths([path.resolve() for path in roots])


def _find_runs_under(root: Path) -> List[Path]:
    if not root.exists():
        return []

    if _is_run_dir(root):
        return [root]

    runs = []
    markers = ['finite_size_scaling.csv', 'threshold_scaling_summary.csv', 'threshold_sweep.csv']
    for marker in markers:
        for marker_path in root.rglob(marker):
            candidate = marker_path.parent
            if _is_run_dir(candidate):
                runs.append(candidate)
    return sorted(_unique_paths(runs))


def _latest_run_from_roots(roots: List[Path]) -> Optional[Path]:
    candidates: List[Path] = []
    for root in roots:
        candidates.extend(_find_runs_under(root))

    if not candidates:
        return None
    return sorted(_unique_paths(candidates))[-1]


def _extract_zip_bytes(zip_name: str, payload: bytes, unpack_root: Path) -> Path:
    unpack_root.mkdir(parents=True, exist_ok=True)
    print(f'Unpacking {zip_name} into {unpack_root} ...')
    with zipfile.ZipFile(io.BytesIO(payload), 'r') as archive:
        archive.extractall(unpack_root)

    run_dir = _latest_run_from_roots([unpack_root])
    if run_dir is None:
        raise FileNotFoundError(
            f'No DSCD run folder found after unpacking {zip_name} into {unpack_root}. '
            'Expected threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv.'
        )
    return run_dir


def _extract_zip_path(zip_path: Path, unpack_root: Path) -> Path:
    zip_path = Path(zip_path).expanduser().resolve()
    if not zip_path.exists():
        raise FileNotFoundError(f'Zip file not found: {zip_path}')

    unpack_root.mkdir(parents=True, exist_ok=True)
    print(f'Unpacking {zip_path} into {unpack_root} ...')
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(unpack_root)

    run_dir = _latest_run_from_roots([unpack_root])
    if run_dir is None:
        raise FileNotFoundError(
            f'No DSCD run folder found after unpacking {zip_path} into {unpack_root}. '
            'Expected threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv.'
        )
    return run_dir


def upload_and_unpack_results_zip() -> Path:
    if colab_files is None:
        raise RuntimeError('Google Colab file picker is unavailable in this environment.')

    print('Pick a DSCD results .zip from your local machine...')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')

    zip_name, payload = next(iter(uploaded.items()))
    if not zip_name.lower().endswith('.zip'):
        raise ValueError(f'Uploaded file is not a .zip: {zip_name}')

    return _extract_zip_bytes(zip_name, payload, _default_unpack_root())




def _ensure_finite_size_scaling_csv(run_dir: Path) -> None:
    target = run_dir / 'finite_size_scaling.csv'
    if target.exists():
        return

    summary_path = run_dir / 'threshold_scaling_summary.csv'
    if not summary_path.exists():
        return

    rows = []
    with summary_path.open('r', newline='') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            num_events = row.get('num_events') or row.get('N')
            width = row.get('transition_width') or row.get('width_0_1_to_0_9')
            max_derivative = row.get('max_derivative') or '0'
            if not num_events or width is None:
                continue
            rows.append({
                'num_events': num_events,
                'transition_width': width,
                'width_0_1_to_0_9': width,
                'max_derivative': max_derivative,
            })

    if not rows:
        return

    with target.open('w', newline='') as fh:
        writer = csv.DictWriter(
            fh,
            fieldnames=['num_events', 'transition_width', 'width_0_1_to_0_9', 'max_derivative'],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f'Generated missing {target.name} from threshold_scaling_summary.csv')


def render_zip_upload_widget() -> None:
    if widgets is None or display is None:
        print('ipywidgets unavailable; use RESULTS_ZIP_PATH or upload_and_unpack_results_zip() in Colab.')
        return

    uploader = widgets.FileUpload(accept='.zip', multiple=False, description='Upload DSCD zip')
    out = widgets.Output()

    def _on_change(change):
        value = uploader.value
        if not value:
            return

        try:
            if isinstance(value, dict):
                item = next(iter(value.values()))
                name = item.get('name', 'uploaded.zip')
                content = item.get('content', b'')
            else:
                item = value[0]
                name = item.get('name', 'uploaded.zip')
                content = item.get('content', b'')

            run_dir = _extract_zip_bytes(name, bytes(content), _default_unpack_root())
            with out:
                out.clear_output()
                print(f'Ready: {run_dir}')
        except Exception as exc:
            with out:
                out.clear_output()
                print(f'Upload failed: {exc}')

    uploader.observe(_on_change, names='value')
    display(uploader, out)


# Show zip picker early on Colab before any auto-generation path.
if _in_colab_runtime() and RUN_DIR is None and RESULTS_ZIP_PATH is None:
    print('Upload a DSCD results .zip (preferred on free Colab):')
    render_zip_upload_widget()


resolved_run_dir: Optional[Path] = None

if RUN_DIR is not None:
    run_dir = Path(RUN_DIR).expanduser().resolve()
    if not _is_run_dir(run_dir):
        raise FileNotFoundError(
            f'Configured RUN_DIR is not a valid DSCD run folder: {run_dir}. '
            'Expected threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv.'
        )
    resolved_run_dir = run_dir
else:
    resolved_run_dir = _latest_run_from_roots(_candidate_output_roots())

if resolved_run_dir is None and RESULTS_ZIP_PATH is not None:
    resolved_run_dir = _extract_zip_path(Path(RESULTS_ZIP_PATH), _default_unpack_root())

if resolved_run_dir is None and colab_files is not None:
    print('No local DSCD run found. Launching Colab zip picker...')
    resolved_run_dir = upload_and_unpack_results_zip()

if resolved_run_dir is None and RESULTS_ZIP_PATH is None and AUTO_BOOTSTRAP_SWEEP:
    if ensure_cargo_available():
        print('No local DSCD run found. Auto-generating from selected dropdown controls...')
        run_sweep_from_controls()
        resolved_run_dir = _latest_run_from_roots(_candidate_output_roots())
    else:
        print('Cargo unavailable; cannot auto-generate DSCD run in this runtime.')

if resolved_run_dir is None:
    searched = '\n'.join(f' - {path}' for path in _candidate_output_roots())
    print('No DSCD outputs found yet.')
    print('Options:')
    print('1) Set RUN_DIR to a run folder with threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv')
    print('2) Set RESULTS_ZIP_PATH to a local .zip path and rerun this cell')
    print('3) In Colab, call upload_and_unpack_results_zip()')
    print('Searched:\n' + searched)
    if colab_files is None:
        render_zip_upload_widget()
    raise FileNotFoundError('No DSCD output folder available for replay.')

_ensure_finite_size_scaling_csv(resolved_run_dir)

RUN_DIR = resolved_run_dir
OUTPUT_ROOT = RUN_DIR.parent
EDGE_PREVIEW_LIMIT = selected_colab_params()['edge_preview_limit']
print(f'Using OUTPUT_ROOT: {OUTPUT_ROOT}')
print(f'Using RUN_DIR: {RUN_DIR}')
RUN_DIR


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

curve_files = sorted(RUN_DIR.glob('threshold_curve_N_*.csv'))
if not curve_files:
    raise FileNotFoundError('No threshold_curve_N_*.csv files found in RUN_DIR')

summary_path = RUN_DIR / 'threshold_scaling_summary.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
else:
    # Backward compatibility for older replay zips without summary.
    finite_path = RUN_DIR / 'finite_size_scaling.csv'
    if finite_path.exists():
        summary = pd.read_csv(finite_path).copy()
        if 'transition_width' not in summary.columns and 'width_0_1_to_0_9' in summary.columns:
            summary['transition_width'] = summary['width_0_1_to_0_9']
        if 'tau_star' not in summary.columns:
            summary['tau_star'] = np.nan
        if 'max_derivative' not in summary.columns:
            summary['max_derivative'] = np.nan
    else:
        summary = pd.DataFrame(columns=['num_events', 'tau_star', 'transition_width', 'max_derivative'])

curve_frames = []
for file in curve_files:
    try:
        n = int(file.stem.split('_')[-1])
    except Exception:
        n = -1
    curve = pd.read_csv(file)
    curve['num_events'] = n
    curve_frames.append(curve)
curves = pd.concat(curve_frames, ignore_index=True)

events_df = pd.read_csv(RUN_DIR / 'graph_events.csv')
edges_df = pd.read_csv(RUN_DIR / 'graph_edges.csv')

def resolve_edge_columns(df):
    candidates = [
        ('src', 'dst'),
        ('source_event_id', 'target_event_id'),
        ('from_event_id', 'to_event_id'),
        ('source', 'target'),
    ]
    for src_col, dst_col in candidates:
        if src_col in df.columns and dst_col in df.columns:
            return src_col, dst_col
    raise KeyError(f'Could not resolve edge columns from: {list(df.columns)}')

src_col, dst_col = resolve_edge_columns(edges_df)

provenance_candidates = [RUN_DIR / 'edge_provenance_0.csv', RUN_DIR / 'edge_provenance.csv']
provenance_path = next((p for p in provenance_candidates if p.exists()), None)
if provenance_path is not None:
    provenance_df = pd.read_csv(provenance_path)
else:
    provenance_df = pd.DataFrame()

print('Loaded curves for N:', sorted(int(n) for n in curves['num_events'].dropna().unique()))
print('Edge columns:', src_col, dst_col)
print('Loaded provenance from:', provenance_path.name if provenance_path is not None else 'not available')
print('EDGE_PREVIEW_LIMIT:', EDGE_PREVIEW_LIMIT)


In [ ]:
# Figure 1: Hero rho(tau) for largest N with tau* marker
largest_n = int(curves['num_events'].max())
hero = curves[curves['num_events'] == largest_n].sort_values('tau')

summary_row = summary[summary['num_events'] == largest_n]
tau_star = float(summary_row['tau_star'].iloc[0]) if not summary_row.empty else np.nan

plt.figure(figsize=(8, 5))
plt.plot(hero['tau'], hero['expansion_ratio'], linewidth=2.5, label=f'N={largest_n}')
if np.isfinite(tau_star):
    plt.axvline(tau_star, color='tab:red', linestyle='--', label=f'tau*={tau_star:.4f}')
plt.xlabel('tau')
plt.ylabel('rho(tau)')
plt.title('DSCD Hero: Expansion Ratio vs Trust Threshold')
plt.grid(alpha=0.25)
plt.legend()
plt.savefig(RUN_DIR / 'fig_dscd_hero_rho_vs_tau.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 2: finite-size scaling
summary = summary.sort_values('num_events')
N = summary['num_events'].to_numpy(dtype=float)
width = summary['transition_width'].to_numpy(dtype=float)
max_derivative = summary['max_derivative'].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].loglog(N, np.maximum(width, 1e-12), marker='o', linewidth=2)
axes[0].set_title('Transition Width vs N')
axes[0].set_xlabel('num_events')
axes[0].set_ylabel('transition_width')
axes[0].grid(alpha=0.25, which='both')

axes[1].loglog(N, np.maximum(max_derivative, 1e-12), marker='o', linewidth=2)
axes[1].set_title('Max Derivative vs N')
axes[1].set_xlabel('num_events')
axes[1].set_ylabel('max_derivative')
axes[1].grid(alpha=0.25, which='both')

plt.tight_layout()
plt.savefig(RUN_DIR / 'fig_dscd_finite_size_scaling.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 3: local DAG preview
def resolve_event_columns(df):
    id_col = next((c for c in ['event_id', 'id'] if c in df.columns), None)
    if id_col is None:
        raise KeyError(f'Could not resolve event id column from: {list(df.columns)}')

    time_col = next((c for c in ['time_index', 'time', 'event_index', 't'] if c in df.columns), None)
    if time_col is None:
        raise KeyError(f'Could not resolve time column from: {list(df.columns)}')

    structural_col = next((c for c in ['structural_level', 'entropy_density', 'echo_slope'] if c in df.columns), None)
    return id_col, time_col, structural_col


event_id_col, time_col, structural_col = resolve_event_columns(events_df)
events_sorted = events_df.sort_values(time_col).reset_index(drop=True)

window_events = min(max(128, EDGE_PREVIEW_LIMIT // 2), len(events_sorted))
start = max(0, (len(events_sorted) // 2) - (window_events // 2))
window_df = events_sorted.iloc[start:start + window_events]
window_ids = set(window_df[event_id_col])

local_edges = edges_df[
    edges_df[src_col].isin(window_ids) & edges_df[dst_col].isin(window_ids)
].copy()

time_map = dict(zip(events_df[event_id_col], events_df[time_col]))
if 'temporal_lag' in local_edges.columns:
    local_edges = local_edges.sort_values('temporal_lag')
else:
    local_edges['temporal_lag_plot'] = (
        local_edges[dst_col].map(time_map) - local_edges[src_col].map(time_map)
    ).abs()
    local_edges = local_edges.sort_values('temporal_lag_plot')

subset_edges = local_edges.head(EDGE_PREVIEW_LIMIT)
if subset_edges.empty:
    fallback_edges = edges_df.copy()
    if 'temporal_lag' in fallback_edges.columns:
        fallback_edges = fallback_edges.sort_values('temporal_lag')
    else:
        fallback_edges['temporal_lag_plot'] = (
            fallback_edges[dst_col].map(time_map) - fallback_edges[src_col].map(time_map)
        ).abs()
        fallback_edges = fallback_edges.sort_values('temporal_lag_plot')
    subset_edges = fallback_edges.head(EDGE_PREVIEW_LIMIT)

subset_nodes = sorted(set(subset_edges[src_col]).union(subset_edges[dst_col]))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[[src_col, dst_col]].itertuples(index=False, name=None))

level_map = (
    dict(zip(events_df[event_id_col], events_df[structural_col]))
    if structural_col is not None
    else {}
)

x_raw = np.array([float(time_map.get(node, 0.0)) for node in graph.nodes()], dtype=float)
if len(x_raw) and np.max(x_raw) > np.min(x_raw):
    x = (x_raw - np.min(x_raw)) / (np.max(x_raw) - np.min(x_raw))
else:
    x = np.zeros_like(x_raw)

if structural_col is not None:
    y_raw = np.array([float(level_map.get(node, 0.0)) for node in graph.nodes()], dtype=float)
    if len(y_raw) and np.max(y_raw) > np.min(y_raw):
        y = (y_raw - np.min(y_raw)) / (np.max(y_raw) - np.min(y_raw))
    else:
        y = np.zeros_like(y_raw)
else:
    order = {node: i for i, node in enumerate(sorted(graph.nodes()))}
    y = np.array([((order[node] % 17) - 8) / 16.0 for node in graph.nodes()], dtype=float)

order = {node: i for i, node in enumerate(sorted(graph.nodes()))}
jitter = np.array([((order[node] % 9) - 4) * 0.01 for node in graph.nodes()], dtype=float)
y = y + jitter

pos = {node: (float(xi), float(yi)) for node, xi, yi in zip(graph.nodes(), x, y)}

plt.figure(figsize=(10, 6))
nx.draw_networkx_edges(graph, pos=pos, alpha=0.35, width=0.7, arrows=True, arrowsize=10)
nx.draw_networkx_nodes(graph, pos=pos, node_size=55, node_color='tab:blue')
plt.title(f'DSCD DAG preview ({len(subset_nodes)} nodes, {len(subset_edges)} local edges)')
plt.xlabel('normalized time')
plt.ylabel(structural_col if structural_col is not None else 'layout rank')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RUN_DIR / 'fig_dscd_dag_preview.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 4: provenance panel
if provenance_df.empty:
    print('No provenance file found (expected edge_provenance_0.csv or edge_provenance.csv). Skipping Figure 4.')
else:
    row = provenance_df.iloc[0]

    def _pick_int(value, default=0):
        try:
            return int(value)
        except Exception:
            return default

    source_id = _pick_int(
        row['source_event'] if 'source_event' in provenance_df.columns else row.get('src', row.get('source_event_id', 0))
    )
    target_id = _pick_int(
        row['target_event'] if 'target_event' in provenance_df.columns else row.get('dst', row.get('target_event_id', 0))
    )

    sub = edges_df[
        ((edges_df[src_col] == source_id) | (edges_df[src_col] == target_id) |
         (edges_df[dst_col] == source_id) | (edges_df[dst_col] == target_id))
    ].head(128)

    if sub.empty:
        print('Provenance edge neighborhood is empty in current edge table; skipping graph panel.')
    else:
        sub_nodes = sorted(set(sub[src_col]).union(sub[dst_col]))

        pg = nx.DiGraph()
        pg.add_nodes_from(sub_nodes)
        pg.add_edges_from(sub[[src_col, dst_col]].itertuples(index=False, name=None))

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        pos = nx.spring_layout(pg, seed=5)
        colors = ['tab:red' if node in (source_id, target_id) else 'tab:blue' for node in pg.nodes()]
        sizes = [220 if node in (source_id, target_id) else 80 for node in pg.nodes()]
        nx.draw_networkx(pg, pos=pos, node_color=colors, node_size=sizes, with_labels=False, arrows=True, ax=axes[0])
        axes[0].set_title('Provenance edge neighborhood')
        axes[0].axis('off')

        axes[1].axis('off')
        keys = [
            'edge_id', 'source_event', 'target_event', 'tau_star', 'base_trust', 'effective_trust',
            'dt', 'd_struct', 'temporal_lag', 'structural_distance',
            'structural_level_source', 'structural_level_target',
            'module_id', 'rewrite_rule_id', 'residual_state_source', 'residual_state_target',
        ]
        lines = []
        for key in keys:
            if key in provenance_df.columns:
                lines.append(f'{key}: {row[key]}')
        axes[1].text(0.02, 0.98, '\n'.join(lines), va='top', family='monospace')
        axes[1].set_title('Edge provenance')

        plt.tight_layout()
        plt.savefig(RUN_DIR / 'fig_dscd_edge_provenance.png', dpi=300, bbox_inches='tight')
        plt.show()
